<a href="https://colab.research.google.com/github/NaruhiShioda/Graduation-Research/blob/main/%E5%8D%92%E6%A5%AD%E7%A0%94%E7%A9%B6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 環境構築とGoogle Driveの連携
import os
import subprocess
from pathlib import Path
from google.colab import drive

# 1. Google Driveとの連携
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

# 2. PyVistaのインストール
try:
    import pyvista as pv
    print("PyVista is already installed.")
except ImportError:
    print("Installing compatible IPython and PyVista...")
    subprocess.run("pip install --upgrade pip numpy scipy", shell=True, check=True)
    subprocess.run("pip install --no-cache-dir --force-reinstall ipython==8.29.0 pyvista", shell=True, check=True)
    print("PyVista installation complete.")

# 3. Dedalus v3のインストール
import importlib.util
if importlib.util.find_spec("dedalus") is not None:
    print("Dedalus v3 is already installed.")
else:
    print("Installing Dedalus v3")
    from IPython import get_ipython
    ipython = get_ipython()
    ipython.system("apt-get update -qq")
    ipython.system("apt-get install -y -qq libfftw3-dev libfftw3-mpi-dev libhdf5-openmpi-dev libopenmpi-dev openmpi-bin")
    os.environ['MPI_INCLUDE_PATH'] = "/usr/lib/x86_64-linux-gnu/openmpi/include"
    os.environ['MPI_LIBRARY_PATH'] = "/usr/lib/x86_64-linux-gnu"
    ipython.system("CC=mpicc pip install -q --no-cache-dir dedalus")
    print("Dedalus v3 installation complete.")

Mounted at /content/drive
Installing compatible IPython and PyVista...
PyVista installation complete.
Installing Dedalus v3
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libfftw3-long3:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../00-libfftw3-long3_3.3.10-1ubuntu3_amd64.deb ...
Unpacking libfftw3-long3:amd64 (3.3.10-1ubuntu3) ...
Selecting previously unselected package libfftw3-quad3:amd64.
Preparing to unpack .../01-libfftw3-quad3_3.3.10-1ubuntu3_amd64.deb ...
Unpacking libfftw3-quad3:amd64 (3.3.10-1ubuntu3) ...
Selecting previously unselected package libfftw3-single3:amd64.
Preparing to unpack .../02-libfftw3-single3_3.3.10-1ubuntu3_amd64.deb ...
Unpacking libfftw3-single3:amd64 (3.3.10-1ubuntu3) ...
Selecting previously unselected package libfftw

In [36]:
%%writefile mpi_worker.py
import os
import sys
import time
import logging
import resource
import argparse
from pathlib import Path
from dataclasses import dataclass
import numpy as np

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['NUMEXPR_MAX_THREADS'] = '1'
os.environ['TRANSFORMS-FFTW_PLANNING_RIGOR'] = 'FFTW_MEASURE'

from mpi4py import MPI
import dedalus.public as d3

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

def get_total_ram_gb() -> float:
    local_ram_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    total_ram_kb = comm.allreduce(local_ram_kb, op=MPI.SUM)
    return total_ram_kb / (1024 ** 2)

@dataclass
class SimConfig:

    Roc: float
    Ra: float

    max_timestep: float
    stop_sim_time: float
    out_dir: Path = Path

    Pr: float = 1.0

    Lx: float = 8.0
    Ly: float = 8.0
    Lz: float = 1.0
    Nx: int = 128
    Ny: int = 128
    Nz: int = 32

    dealias: float = 1.5

    amplitude: float = 1.0
    R0: float = 1.0
    AR: float = 1.5

    @property
    def Nu(self) -> float:
        return float(np.sqrt(self.Pr / self.Ra))
    @property
    def Ka(self) -> float:
        return self.Nu / self.Pr
    @property
    def Ta(self) -> float:
        return self.Ra / (self.Pr * (self.Roc ** 2))
    @property
    def Fc(self) -> float:
        return (float(np.sqrt(self.Ta)) * self.Nu) / (self.Lz ** 2)
    @property
    def Rx(self) -> float:
        return self.R0 * float(np.sqrt(self.AR))
    @property
    def Ry(self) -> float:
        return self.R0 / float(np.sqrt(self.AR))
    @property
    def xcen(self) -> float:
        return self.Lx / 2.0
    @property
    def ycen(self) -> float:
        return self.Ly / 2.0

def setup_rank0_logger(log_file: Path) -> logging.Logger:
    logger = logging.getLogger("MPIWorker")
    logger.propagate = False
    logger.handlers.clear()

    if rank == 0:
        logger.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s [MPI:%(levelname)s] %(message)s', datefmt='%H:%M:%S')
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(formatter)
        logger.addHandler(sh)
        fh = logging.FileHandler(log_file, encoding='utf-8')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
    else:
        logger.setLevel(logging.ERROR)
    return logger

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--ra', type=float, required=True)
    parser.add_argument('--roc', type=float, required=True)
    parser.add_argument('--ar', type=float, required=True)
    parser.add_argument('--max_dt', type=float, required=True)
    parser.add_argument('--sim_time', type=float, required=True)
    parser.add_argument('--out_dir', type=str, required=True)
    args = parser.parse_args()

    out_dir = Path(args.out_dir)
    if rank == 0:
        out_dir.mkdir(parents=True, exist_ok=True)
    comm.Barrier()

    cfg = SimConfig(
        Ra=args.ra, Roc=args.roc, AR=args.ar,
        max_timestep=args.max_dt, stop_sim_time=args.sim_time, out_dir=out_dir
    )
    logger = setup_rank0_logger(out_dir / "simulation.log")

    logging.getLogger('subsystems').setLevel(logging.WARNING)
    logging.getLogger('solvers').setLevel(logging.WARNING)

    logger.info(f"=== MPI Parallel Execution Started with {size} Cores ===")
    logger.info(f"Parameters: Ra={cfg.Ra:.1e}, Roc={cfg.Roc}, AR={cfg.AR}, Nu={cfg.Nu:.2e}, Fc={cfg.Fc:.2e}")
    logger.info("Building bases, fields, and FFTW plans...")
    if rank == 0: sys.stdout.flush()

    coords = d3.CartesianCoordinates('x', 'y', 'z')
    dist = d3.Distributor(coords, dtype=np.float64)
    xbasis = d3.RealFourier(coords['x'], size=cfg.Nx, bounds=(0, cfg.Lx), dealias=cfg.dealias)
    ybasis = d3.RealFourier(coords['y'], size=cfg.Ny, bounds=(0, cfg.Ly), dealias=cfg.dealias)
    zbasis = d3.ChebyshevT(coords['z'], size=cfg.Nz, bounds=(0, cfg.Lz), dealias=cfg.dealias)
    x, y, z = dist.local_grids(xbasis, ybasis, zbasis)

    p = dist.Field(name='p', bases=(xbasis, ybasis, zbasis))
    b = dist.Field(name='b', bases=(xbasis, ybasis, zbasis))
    b_bottom = dist.Field(name='b_bottom', bases=(xbasis, ybasis))
    u = dist.VectorField(coords, name='u', bases=(xbasis, ybasis, zbasis))

    tau_p = dist.Field(name='tau_p')
    tau_b1 = dist.Field(name='tau_b1', bases=(xbasis, ybasis))
    tau_b2 = dist.Field(name='tau_b2', bases=(xbasis, ybasis))
    tau_u1 = dist.VectorField(coords, name='tau_u1', bases=(xbasis, ybasis))
    tau_u2 = dist.VectorField(coords, name='tau_u2', bases=(xbasis, ybasis))

    ex, ey, ez = coords.unit_vector_fields(dist)
    lift_basis = zbasis.derivative_basis(1)
    lift = lambda A: d3.Lift(A, lift_basis, -1)
    dz = lambda A: d3.Differentiate(A, coords['z'])

    grad_u = d3.grad(u) + ez * lift(tau_u1)
    grad_b = d3.grad(b) + ez * lift(tau_b1)
    coriolis_vec = - (u @ ey) * ex + (u @ ex) * ey

    b_bottom['g'] = cfg.amplitude * np.exp(
        - (((x - cfg.xcen) ** 2 / (cfg.Rx ** 2)) + ((y - cfg.ycen) ** 2 / (cfg.Ry ** 2)))
    )

    problem = d3.IVP([p, b, u, tau_p, tau_b1, tau_b2, tau_u1, tau_u2], namespace=locals())
    problem.add_equation("trace(grad_u) + tau_p = 0")
    problem.add_equation(f"dt(b) - {cfg.Ka}*div(grad_b) + lift(tau_b2) = - u@grad(b)")
    problem.add_equation(f"dt(u) - {cfg.Nu}*div(grad_u) + grad(p) - b*ez + lift(tau_u2) = - u@grad(u) - {cfg.Fc}*coriolis_vec")

    problem.add_equation("b(z=0) = b_bottom")
    problem.add_equation(f"b(z={cfg.Lz}) = 0")
    problem.add_equation("integ(p) = 0")
    problem.add_equation("ez@u(z=0) = 0")
    problem.add_equation("ex@dz(u)(z=0) = 0")
    problem.add_equation("ey@dz(u)(z=0) = 0")
    problem.add_equation(f"ez@u(z={cfg.Lz}) = 0")
    problem.add_equation(f"ex@dz(u)(z={cfg.Lz}) = 0")
    problem.add_equation(f"ey@dz(u)(z={cfg.Lz}) = 0")

    solver = problem.build_solver(d3.RK222)
    solver.stop_sim_time = cfg.stop_sim_time
    total_ram = get_total_ram_gb()
    logger.info(f"Solver built successfully! (Total MPI RAM: {total_ram:.2f} GB)")
    if rank == 0: sys.stdout.flush()

    vorticity = d3.curl(u)
    vort_z = d3.dot(vorticity, ez)
    w = d3.dot(u, ez)
    u_horizon = u - w * ez
    kE_2D = 0.5 * d3.dot(u_horizon, u_horizon)
    kE_3D = 0.5 * (w * w)

    b.fill_random('g', seed=42, distribution='normal', scale=1e-3)
    b['g'] *= z * (cfg.Lz - z)
    b['g'] += b_bottom['g'] * (cfg.Lz - z) / cfg.Lz
    initial_dt = 1e-2

    snapshots = solver.evaluator.add_file_handler(out_dir / 'snapshots', sim_dt=2.0, max_writes=10, mode='overwrite')
    snapshots.add_task(b, name='buoyancy')
    snapshots.add_task(u, name='velocity')
    snapshots.add_task(vorticity, name='vorticity')

    timeseries = solver.evaluator.add_file_handler(out_dir / 'timeseries', sim_dt=0.2, max_writes=250, mode='overwrite')
    timeseries.add_task(d3.Integrate(w * b), name='wb_buoyancy_flux')
    timeseries.add_task(d3.Integrate(kE_2D), name='KE_2D')
    timeseries.add_task(d3.Integrate(kE_3D), name='KE_3D')
    timeseries.add_task(d3.Integrate(w * dz(p)), name='pressure_work')
    timeseries.add_task(d3.Integrate(vort_z * vort_z), name='Enstrophy_Z')
    timeseries.add_task(d3.Integrate(cfg.Nu * d3.dot(vorticity, vorticity)), name='viscous_dissipation')

    checkpoints = solver.evaluator.add_file_handler(out_dir / 'checkpoints', sim_dt=20.0, max_writes=1, mode='overwrite')
    checkpoints.add_tasks(solver.state)

    CFL = d3.CFL(
        solver, initial_dt=initial_dt, cadence=10, safety=0.35, threshold=0.05,
        max_change=1.5, min_change=0.2, max_dt=cfg.max_timestep
    )
    CFL.add_velocity(u)

    flow = d3.GlobalFlowProperty(solver, cadence=10)
    flow.add_property((u @ u) ** 0.5 / cfg.Nu, name='Re')
    flow.add_property(d3.abs(w), name='abs_w')

    try:
        logger.info("Starting main time loop...")
        if rank == 0: sys.stdout.flush()
        start_wall_time = time.perf_counter()

        while solver.proceed:
            timestep = CFL.compute_timestep()
            solver.step(timestep)

            if (solver.iteration - 1) % 10 == 0:
                max_Re = flow.max('Re')
                max_w = flow.max('abs_w')
                ram_gb = get_total_ram_gb()

                if rank == 0:
                    progress_pct = (solver.sim_time / cfg.stop_sim_time) * 100.0
                    elapsed_sec = time.perf_counter() - start_wall_time
                    if solver.sim_time > 1e-5:
                        eta_sec = elapsed_sec * ((cfg.stop_sim_time - solver.sim_time) / solver.sim_time)
                        eta_str = f"{int(eta_sec // 60):02d}m{int(eta_sec % 60):02d}s"
                    else:
                        eta_str = "--m--s"

                    logger.info(
                        f"[{progress_pct:5.1f}% | ETA {eta_str} | RAM {ram_gb:4.1f}GB] "
                        f"Iter={solver.iteration:05d} | Time={solver.sim_time:6.2f}/{cfg.stop_sim_time:.0f} | "
                        f"dt={timestep:.2e} | Re={max_Re:6.1f} | max|w|={max_w:.2e}"
                    )
                    sys.stdout.flush()

                if np.isnan(max_Re):
                    logger.error("物理量がNaNになり発散しました。")
                    sys.exit(1)
    finally:
        solver.log_stats()

if __name__ == "__main__":
    main()

Overwriting mpi_worker.py


In [32]:
%%writefile sweep_master.py
import os
import sys
import time
import shutil
import datetime
import subprocess
import gc
import re
import multiprocessing as mp
from itertools import product
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import h5py

# ==============================================================================
# 1. コア数の自動判定
# ==============================================================================
def get_safe_mpi_ranks() -> int:
    total_threads = os.cpu_count() or 2
    physical_est = max(1, total_threads // 2)
    if physical_est >= 4:
        return 4
    elif physical_est >= 2:
        return 2
    return 1


# ==============================================================================
# 2. マルチプロセス並列VTK変換（H5DSエラー完全回避版）
# ==============================================================================
def _extract_grid_coords(f: h5py.File, b_shape: tuple):
    """dims APIを使わず、scalesグループから安全に(x, y, z)座標配列を取得する"""
    _, nx, ny, nz = b_shape
    x, y, z = None, None, None

    if 'scales' in f:
        for key in f['scales'].keys():
            if key.startswith('x_hash') or key == 'x':
                x = f['scales'][key][:]
            elif key.startswith('y_hash') or key == 'y':
                y = f['scales'][key][:]
            elif key.startswith('z_hash') or key == 'z':
                z = f['scales'][key][:]

    if x is None or len(x) != nx:
        x = np.linspace(0, 8.0, nx, endpoint=False)
    if y is None or len(y) != ny:
        y = np.linspace(0, 8.0, ny, endpoint=False)
    if z is None or len(z) != nz:
        j = np.arange(nz)
        z = 0.5 * (1.0 - np.cos(np.pi * j / (nz - 1 if nz > 1 else 1)))

    return x, y, z


def _process_single_h5_to_vtk(args) -> int:
    """1つのHDF5ファイルを受け取り、指定の開始番号から連番でVTKを書き出すワーカー関数"""
    h5_file, vtk_dir, start_idx = args
    import pyvista as pv

    written = 0
    with h5py.File(h5_file, mode='r') as f:
        if 'tasks' not in f or 'buoyancy' not in f['tasks']:
            return 0
        b_ds = f['tasks']['buoyancy']
        u_ds = f['tasks']['velocity']
        v_ds = f['tasks']['vorticity']

        n_frames = b_ds.shape[0]
        x, y, z = _extract_grid_coords(f, b_ds.shape)

        grid = pv.RectilinearGrid(x, y, z)

        for i in range(n_frames):
            grid.point_data['buoyancy'] = b_ds[i].reshape(-1, order='F')
            grid.point_data['velocity'] = u_ds[i].reshape(3, -1, order='F').T
            grid.point_data['vorticity'] = v_ds[i].reshape(3, -1, order='F').T

            frame_num = start_idx + i
            grid.save(vtk_dir / f"snapshot_{frame_num:04d}.vtr")
            written += 1

        del grid
    gc.collect()
    return written


def convert_to_vtk_parallel(h5_dir: Path, vtk_dir: Path, num_workers: int):
    t0 = time.perf_counter()
    vtk_dir.mkdir(parents=True, exist_ok=True)

    def natural_keys(text):
        return [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', str(text))]

    snapshots_dir = h5_dir / 'snapshots'
    if not snapshots_dir.exists():
        print(f"⚠️ Snapshotsフォルダが見つかりません: {snapshots_dir}", flush=True)
        return

    # ★修正: rglobではなくglobを使い、サブフォルダ内のMPI断片ファイル(_p0.h5等)を除外して親ファイルのみ取得
    h5_files = sorted(snapshots_dir.glob("*.h5"), key=lambda p: natural_keys(p.name))
    if not h5_files:
        # もし親ファイルがサブフォルダにある構成だった場合のみ、_p数字.h5 を除外して検索
        h5_files = sorted(
            [p for p in snapshots_dir.rglob("*.h5") if not re.search(r'_p\d+\.h5$', p.name)],
            key=lambda p: natural_keys(p.name)
        )

    if not h5_files:
        print(" 変換対象のHDF5ファイルが見つかりません。", flush=True)
        return

    tasks = []
    current_offset = 0
    for h5_file in h5_files:
        with h5py.File(h5_file, mode='r') as f:
            if 'tasks' not in f or 'buoyancy' not in f['tasks']:
                continue
            n_frames = f['tasks']['buoyancy'].shape[0]
        tasks.append((h5_file, vtk_dir, current_offset))
        current_offset += n_frames

    total_frames = current_offset
    print(f" 全 {total_frames} フレーム ({len(tasks)} ファイル) を {num_workers} コア並列でVTKへ変換しています...", flush=True)

    # ★修正: mp_context=mp.get_context('spawn') を指定してHDF5のプロセス間ロック競合を完全防止
    completed_frames = 0
    ctx = mp.get_context('spawn')
    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        futures = [executor.submit(_process_single_h5_to_vtk, t) for t in tasks]
        for future in as_completed(futures):
            completed_frames += future.result()
            print(f"  -> 並列VTK変換進捗: {completed_frames}/{total_frames} フレーム完了", flush=True)

    elapsed = time.perf_counter() - t0
    print(f"✅ 全 {total_frames} フレームのVTK変換が {elapsed:.1f} 秒で完了しました！", flush=True)


# ==============================================================================
# 3. Google Driveへのバックアップと完了判定
# ==============================================================================
def backup_data(base_dir: Path, drive_dir: Path, out_dir: Path, vtk_dir: Path, folder_name: str):
    print(" Google Driveへデータをバックアップしています...", flush=True)
    target_dir = drive_dir / folder_name
    target_dir.mkdir(parents=True, exist_ok=True)

    if vtk_dir.exists() and any(vtk_dir.iterdir()):
        zip_base = base_dir / f"{folder_name}_VTK"
        shutil.make_archive(str(zip_base), 'zip', str(vtk_dir))
        shutil.copy2(f"{zip_base}.zip", target_dir)

    shutil.copytree(out_dir, target_dir / "h5_data", dirs_exist_ok=True)
    print(f"✅ バックアップ完了！ 保存先: {target_dir}", flush=True)


# ==============================================================================
# 4. メインループ（パラメータスイープの実行管理）
# ==============================================================================
if __name__ == "__main__":
    ra_list = [1.0e4]
    roc_list = [1.5]
    ar_list = [1.5]
    sim_time = 100.0

    base_dir = Path('/content')
    drive_dir = Path('/content/drive/MyDrive/dedalus_data')
    num_cores = get_safe_mpi_ranks()

    sweep_combinations = list(product(ra_list, roc_list, ar_list))
    total_runs = len(sweep_combinations)

    print(f"🚀 MPI {num_cores} 並列モードで全 {total_runs} パターンのスイープを開始します。", flush=True)
    print("=" * 65, flush=True)

    for idx, (ra, roc, ar) in enumerate(sweep_combinations, start=1):
        max_dt = 0.25

        JST = datetime.timezone(datetime.timedelta(hours=+9), 'JST')
        now = datetime.datetime.now(JST).strftime("%Y%m%d_%H%M%S")
        folder_name = f"{now}_Ra{ra:.1e}_Roc{roc:.1e}_AR{ar:.1e}"
        out_dir = base_dir / folder_name
        vtk_dir = base_dir / f"VTK_{folder_name}"

        print(f"\n▶️ [{idx}/{total_runs}] Starting MPI Run ({num_cores} cores): Ra={ra:.1e}, Roc={roc}, AR={ar}", flush=True)

        cmd = [
            "mpiexec", "--allow-run-as-root", "--oversubscribe",
            "-n", str(num_cores),
            "python3", "-u", "mpi_worker.py",
            "--ra", str(ra),
            "--roc", str(roc),
            "--ar", str(ar),
            "--max_dt", str(max_dt),
            "--sim_time", str(sim_time),
            "--out_dir", str(out_dir)
        ]

        try:
            res = subprocess.run(cmd)
            if res.returncode != 0:
                print(f"⚠️ [{idx}/{total_runs}] MPI計算が異常終了しました (Code: {res.returncode})。次へ進みます。", flush=True)
                continue

            convert_to_vtk_parallel(out_dir, vtk_dir, num_workers=num_cores)
            backup_data(base_dir, drive_dir, out_dir, vtk_dir, folder_name)
            print(f"✅ [{idx}/{total_runs}] Completed: Ra={ra:.1e}, Roc={roc}, AR={ar}", flush=True)

        finally:
            if out_dir.exists():
                shutil.rmtree(out_dir, ignore_errors=True)
            if vtk_dir.exists():
                shutil.rmtree(vtk_dir, ignore_errors=True)
            zip_file = base_dir / f"{folder_name}_VTK.zip"
            if zip_file.exists():
                zip_file.unlink()
            gc.collect()
            print("-" * 65, flush=True)

    print("🎉 すべてのMPIパラメータスイープ処理が完了しました！", flush=True)

Overwriting sweep_master.py


In [ ]:
!python3 -u sweep_master.py

🚀 MPI 4 並列モードで全 1 パターンのスイープを開始します。

▶️ [1/1] Starting MPI Run (4 cores): Ra=1.0e+04, Roc=1.5, AR=1.5
06:11:53 [MPI:INFO] === MPI Parallel Execution Started with 4 Cores ===
06:11:53 [MPI:INFO] Parameters: Ra=1.0e+04, Roc=1.5, AR=1.5, Nu=1.00e-02, Fc=6.67e-01
06:11:53 [MPI:INFO] Building bases, fields, and FFTW plans...
06:13:18 [MPI:INFO] Solver built successfully! (Total MPI RAM: 1.60 GB)
06:13:18 [MPI:INFO] Starting main time loop...
06:13:30 [MPI:INFO] [  0.0% | ETA 1993m35s | RAM  8.5GB] Iter=00001 | Time=  0.01/100 | dt=1.00e-02 | Re=   0.0 | max|w|=0.00e+00
06:13:44 [MPI:INFO] [  0.1% | ETA 386m40s | RAM  9.0GB] Iter=00011 | Time=  0.11/100 | dt=1.00e-02 | Re=   2.3 | max|w|=1.59e-02
06:14:03 [MPI:INFO] [  0.3% | ETA 285m34s | RAM 11.4GB] Iter=00021 | Time=  0.26/100 | dt=1.50e-02 | Re=   5.4 | max|w|=3.85e-02
06:14:21 [MPI:INFO] [  0.5% | ETA 216m59s | RAM 13.0GB] Iter=00031 | Time=  0.49/100 | dt=2.25e-02 | Re=   9.5 | max|w|=7.16e-02
06:14:39 [MPI:INFO] [  0.8% | ETA 162m40s |